In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

In [2]:
DATA_PATH = r"ChestXRayPneumonia"
train_dir = os.path.join(DATA_PATH, "train")
val_dir = os.path.join(DATA_PATH, "val")
test_dir = os.path.join(DATA_PATH, "test")

In [3]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

In [4]:
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary'
)

val_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary'
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=(150,150),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

Found 5216 images belonging to 2 classes.
Found 16 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


In [5]:
model = models.Sequential([
    # 1st layer
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150,150,3)),
    layers.MaxPooling2D((2,2)),

    # 2nd layer
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    # 3rd layer
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    # Flatten  feature map to 1D vector
    layers.Flatten(),

    # Fully connected dense layer
    layers.Dense(128, activation='relu'),

    # Output layer: binary classification
    layers.Dense(1, activation='sigmoid')
])

C:\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [6]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36992)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,735,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,828,481 (18.42 MB)

 Trainable params: 4,828,481 (18.42 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [8]:
# -------------------------------
# Step 6: Train Model
# -------------------------------
# Fit the model with training generator and validate with validation generator
history = model.fit(
    train_generator,                  # training data generator
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    # number of batches per epoch = total samples / batch size
    epochs=10,                        # number of full passes over training data
    validation_data=val_generator,    # validation data generator
    validation_steps=val_generator.samples // val_generator.batch_size
    # number of batches for validation
)

Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 142s 857ms/step - accuracy: 0.7782 - loss: 0.4608 - val_accuracy: 0.7500 - val_loss: 0.7203
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 119s 731ms/step - accuracy: 0.8819 - loss: 0.2749 - val_accuracy: 0.6250 - val_loss: 0.8938
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 117s 717ms/step - accuracy: 0.9166 - loss: 0.2152 - val_accuracy: 0.6875 - val_loss: 0.6564
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 118s 722ms/step - accuracy: 0.9233 - loss: 0.1915 - val_accuracy: 0.8125 - val_loss: 0.3761
Epoch 5/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 121s 744ms/step - accuracy: 0.9304 - loss: 0.1781 - val_accuracy: 0.8125 - val_loss: 0.5510
Epoch 6/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 116s 710ms/step - accuracy: 0.9331 - loss: 0.1713 - val_accuracy: 0.8125 - val_loss: 0.4700
Epoch 7/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 119s 731ms/step - accuracy: 0.9348 - loss: 0.1644 - val_accuracy: 0.6875 - val_loss: 0.4758
Epoch 8/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 121s 742ms/step - accuracy: 0.9461 -

In [9]:
# Evaluate performance on test set
test_loss, test_acc = model.evaluate(
    test_generator,
    steps=test_generator.samples // test_generator.batch_size
)
print("\nTest accuracy:", test_acc)  # print test accuracy

19/19 ━━━━━━━━━━━━━━━━━━━━ 9s 454ms/step - accuracy: 0.7845 - loss: 0.6205

Test accuracy: 0.7845394611358643
